# ResNet-18 CIFAR-10 Teacher vs MLP Students

Load saved probability artifacts for the CIFAR-10 test split and OOD datasets,
then compare teacher and student probability distributions for the saved student distillation methods.
Far-OOD groups MNIST and SVHN, while near-OOD corresponds to CIFAR-100.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch

sns.set_theme(style="darkgrid", context="notebook")

ROOT = Path.cwd()
if not (ROOT / "runs").exists():
    ROOT = ROOT.parent

SRC_PATH = ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from distill_ood_detection.evaluation.ood_scores import (
    absolute_max_probability_difference,
    logit_l2_distance,
    max_probability_difference,
    student_teacher_kl_divergence,
)

EXPERIMENT_NAME = "mlp_student_resnet18_cifar10"
PROBABILITY_DIR = ROOT / "runs" / EXPERIMENT_NAME / "probabilities"
MANIFEST_PATH = PROBABILITY_DIR / "manifest.json"
CHECKPOINT = "best"

METHODS = {
    "mse_softmax": "MSE Softmax",
    "cross_entropy": "Cross-Entropy",
    "mse_logits": "MSE Logits",
}
DATASETS = {
    "cifar10_test": "ID: CIFAR-10 test",
    "mnist_test": "Far-OOD: MNIST test",
    "svhn_test": "Far-OOD: SVHN test",
    "cifar100_test": "Near-OOD: CIFAR-100 test",
}
DATASET_CLASS_NAMES = {
    "cifar10_test": [
        "airplane",
        "automobile",
        "bird",
        "cat",
        "deer",
        "dog",
        "frog",
        "horse",
        "ship",
        "truck",
    ],
    "mnist_test": [str(index) for index in range(10)],
    "svhn_test": [str(index) for index in range(10)],
    "cifar100_test": [str(index) for index in range(10)],
}

MANIFEST_PATH


In [ ]:
with MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    manifest = json.load(handle)

manifest["experiment_name"], manifest["checkpoint_selection"]


In [ ]:
def load_probability_artifact(path: str | Path) -> dict[str, object]:
    """Load one saved probability artifact on CPU."""

    artifact_path = Path(path)
    if not artifact_path.is_absolute():
        artifact_path = ROOT / artifact_path
    return torch.load(artifact_path, map_location="cpu", weights_only=False)


artifacts = []
for entry in manifest["artifacts"]:
    artifact = load_probability_artifact(entry["path"])
    artifacts.append({**artifact, **entry})

len(artifacts)


In [ ]:
teacher_probabilities = {
    artifact["dataset"]: artifact
    for artifact in artifacts
    if artifact["model"] == "teacher"
}

student_probabilities = {
    (artifact["dataset"], artifact["method"], artifact["checkpoint"]): artifact
    for artifact in artifacts
    if artifact["model"] == "student"
}

available_methods = sorted(
    {
        artifact["method"]
        for artifact in artifacts
        if artifact["model"] == "student" and artifact["checkpoint"] == CHECKPOINT
    },
    key=list(METHODS).index,
)
available_datasets = sorted(
    teacher_probabilities,
    key=list(DATASETS).index,
)

if not available_methods:
    raise ValueError(
        f"No student probability artifacts found for checkpoint={CHECKPOINT!r} in {MANIFEST_PATH}."
    )

METHOD_ORDER = available_methods
DATASET_ORDER = [
    dataset_key
    for dataset_key in DATASETS
    if dataset_key in available_datasets
    and any(
        (dataset_key, method_key, CHECKPOINT) in student_probabilities
        for method_key in METHOD_ORDER
    )
]


def to_numpy(tensor: torch.Tensor) -> np.ndarray:
    """Move a tensor to CPU and convert it to a NumPy array."""

    return tensor.detach().cpu().numpy()


def build_argmax_heatmap(
    teacher_probs: np.ndarray,
    student_probs: np.ndarray,
    num_classes: int,
) -> np.ndarray:
    """Compute row-normalized teacher-vs-student argmax percentages."""

    teacher_argmax = teacher_probs.argmax(axis=1)
    student_argmax = student_probs.argmax(axis=1)

    counts = np.zeros((num_classes, num_classes), dtype=np.int64)
    np.add.at(counts, (teacher_argmax, student_argmax), 1)

    row_totals = counts.sum(axis=1, keepdims=True)
    return np.divide(
        counts * 100.0,
        row_totals,
        out=np.zeros_like(counts, dtype=float),
        where=row_totals > 0,
    )


def build_comparison(dataset_key: str, method_key: str) -> dict[str, object]:
    """Assemble teacher/student comparison data for one dataset and method."""

    teacher = teacher_probabilities[dataset_key]
    student = student_probabilities[(dataset_key, method_key, CHECKPOINT)]

    teacher_logits = to_numpy(teacher["logits"])
    student_logits = to_numpy(student["logits"])
    teacher_probs = to_numpy(teacher["probabilities"])
    student_probs = to_numpy(student["probabilities"])
    labels = to_numpy(teacher["labels"]).astype(int)
    sample_indices = np.arange(labels.shape[0])
    teacher_minus_student = teacher_probs - student_probs

    comparison = {
        "teacher_logits": teacher_logits,
        "student_logits": student_logits,
        "teacher_probs": teacher_probs,
        "student_probs": student_probs,
        "labels": labels,
        "predicted_values": pd.DataFrame(
            {
                "probability": np.concatenate(
                    [teacher_probs.ravel(), student_probs.ravel()]
                ),
                "model": ["Teacher"] * teacher_probs.size
                + ["Student"] * student_probs.size,
            }
        ),
        "max_probability_difference": max_probability_difference(
            teacher_probs,
            student_probs,
        ),
        "absolute_max_probability_difference": absolute_max_probability_difference(
            teacher_probs,
            student_probs,
        ),
        "student_teacher_kl_divergence": student_teacher_kl_divergence(
            teacher_probs,
            student_probs,
        ),
        "logit_l2_distance": logit_l2_distance(
            teacher_logits,
            student_logits,
        ),
        "argmax_agreement": build_argmax_heatmap(
            teacher_probs,
            student_probs,
            num_classes=teacher_probs.shape[1],
        ),
    }

    valid_label_mask = labels < teacher_probs.shape[1]
    if np.any(valid_label_mask):
        valid_labels = labels[valid_label_mask]
        valid_indices = sample_indices[valid_label_mask]
        class_names = DATASET_CLASS_NAMES[dataset_key]
        comparison["per_class_delta"] = pd.DataFrame(
            {
                "class_name": [class_names[label] for label in valid_labels],
                "teacher_minus_student": teacher_minus_student[
                    valid_indices, valid_labels
                ],
            }
        )

    return comparison


comparisons = {
    (method_key, dataset_key): build_comparison(dataset_key, method_key)
    for method_key in METHOD_ORDER
    for dataset_key in DATASET_ORDER
    if (dataset_key, method_key, CHECKPOINT) in student_probabilities
}


def dataset_group(dataset_key: str) -> str:
    """Map each dataset to the ID/far-OOD/near-OOD grouping used in the plots."""

    if dataset_key == "cifar10_test":
        return "ID"
    if dataset_key == "cifar100_test":
        return "Near-OOD"
    return "Far-OOD"


DISTRIBUTION_ORDER = ["ID", "Far-OOD", "Near-OOD"]
DISTRIBUTION_CLASS_NAMES = {
    "ID": DATASET_CLASS_NAMES["cifar10_test"],
    "Far-OOD": DATASET_CLASS_NAMES["mnist_test"],
    "Near-OOD": DATASET_CLASS_NAMES["cifar100_test"],
}


def build_probability_group_frame() -> pd.DataFrame:
    """Collect probability values into one dataframe by method and group."""

    probability_frames = []
    for (method_key, dataset_key), comparison in comparisons.items():
        probabilities = np.concatenate(
            [
                comparison["teacher_probs"].ravel(),
                comparison["student_probs"].ravel(),
            ]
        )
        probability_frames.append(
            pd.DataFrame(
                {
                    "probability": probabilities,
                    "distribution": dataset_group(dataset_key),
                    "dataset": DATASETS[dataset_key],
                    "method": METHODS[method_key],
                }
            )
        )

    return pd.concat(probability_frames, ignore_index=True)


def build_grouped_metric_frame(
    metric_key: str,
    value_name: str,
) -> pd.DataFrame:
    """Collect one comparison metric into a dataframe by method and group."""

    metric_frames = []
    for (method_key, dataset_key), comparison in comparisons.items():
        metric_frames.append(
            pd.DataFrame(
                {
                    value_name: comparison[metric_key],
                    "distribution": dataset_group(dataset_key),
                    "dataset": DATASETS[dataset_key],
                    "method": METHODS[method_key],
                }
            )
        )

    return pd.concat(metric_frames, ignore_index=True)


def build_grouped_argmax_heatmaps() -> dict[tuple[str, str], np.ndarray]:
    """Aggregate teacher-vs-student argmax agreement by method and group."""

    heatmaps = {}
    for method_key in METHOD_ORDER:
        for distribution in DISTRIBUTION_ORDER:
            grouped_comparisons = [
                comparison
                for (candidate_method_key, dataset_key), comparison in comparisons.items()
                if candidate_method_key == method_key
                and dataset_group(dataset_key) == distribution
            ]
            if not grouped_comparisons:
                continue
            teacher_probs = np.concatenate(
                [comparison["teacher_probs"] for comparison in grouped_comparisons],
                axis=0,
            )
            student_probs = np.concatenate(
                [comparison["student_probs"] for comparison in grouped_comparisons],
                axis=0,
            )
            heatmaps[(method_key, distribution)] = build_argmax_heatmap(
                teacher_probs,
                student_probs,
                num_classes=len(DISTRIBUTION_CLASS_NAMES[distribution]),
            )

    return heatmaps


def build_grouped_per_class_delta_frames() -> dict[tuple[str, str], pd.DataFrame]:
    """Aggregate ground-truth class deltas by method and group."""

    grouped_frames = {}
    for method_key in METHOD_ORDER:
        for distribution in DISTRIBUTION_ORDER:
            frames = [
                comparison["per_class_delta"]
                for (candidate_method_key, dataset_key), comparison in comparisons.items()
                if candidate_method_key == method_key
                and dataset_group(dataset_key) == distribution
                and "per_class_delta" in comparison
            ]
            if not frames:
                continue
            grouped_frames[(method_key, distribution)] = pd.concat(
                frames,
                ignore_index=True,
            )

    return grouped_frames


probability_group_frame = build_probability_group_frame()
max_probability_difference_group_frame = build_grouped_metric_frame(
    metric_key="max_probability_difference",
    value_name="max_probability_difference",
)
absolute_max_probability_difference_group_frame = build_grouped_metric_frame(
    metric_key="absolute_max_probability_difference",
    value_name="absolute_max_probability_difference",
)
student_teacher_kl_divergence_group_frame = build_grouped_metric_frame(
    metric_key="student_teacher_kl_divergence",
    value_name="student_teacher_kl_divergence",
)
logit_l2_distance_group_frame = build_grouped_metric_frame(
    metric_key="logit_l2_distance",
    value_name="logit_l2_distance",
)
grouped_argmax_heatmaps = build_grouped_argmax_heatmaps()
grouped_per_class_delta_frames = build_grouped_per_class_delta_frames()

{
    "available_methods": [METHODS[method_key] for method_key in METHOD_ORDER],
    "available_datasets": [DATASETS[dataset_key] for dataset_key in DATASET_ORDER],
    "comparison_shapes": {
        (method_key, dataset_key): {
            "teacher_logits_shape": comparison["teacher_logits"].shape,
            "student_logits_shape": comparison["student_logits"].shape,
            "teacher_probs_shape": comparison["teacher_probs"].shape,
            "student_probs_shape": comparison["student_probs"].shape,
        }
        for (method_key, dataset_key), comparison in comparisons.items()
    },
}


The histograms below compare maximum class probabilities for the teacher and students by dataset group.


In [ ]:
def make_axes_grid(
    figsize: tuple[int, int] | None = None,
) -> tuple[plt.Figure, np.ndarray]:
    """Create a grid with one subplot per available method."""

    method_count = len(METHOD_ORDER)
    ncols = method_count
    nrows = int(np.ceil(method_count / ncols))
    if figsize is None:
        figsize = (7 * ncols, 5 * nrows)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()
    for ax in axes[method_count:]:
        ax.set_visible(False)
    return fig, axes


probability_bins = np.linspace(0.0, 1.0, 41)

deep_palette = sns.color_palette("deep")
dark_palette = sns.color_palette("dark")
distribution_palette = {
    "ID": deep_palette[0],
    "Far-OOD": deep_palette[3],
    "Near-OOD": dark_palette[2],
}


def build_max_probability_group_frame(
    probability_key: str,
    value_name: str,
) -> pd.DataFrame:
    """Collect maximum probabilities into one dataframe by method and group."""

    probability_frames = []
    for (method_key, dataset_key), comparison in comparisons.items():
        probability_frames.append(
            pd.DataFrame(
                {
                    value_name: comparison[probability_key].max(axis=1),
                    "distribution": dataset_group(dataset_key),
                    "dataset": DATASETS[dataset_key],
                    "method": METHODS[method_key],
                }
            )
        )

    return pd.concat(probability_frames, ignore_index=True)


def plot_max_probability_histograms(
    frame: pd.DataFrame,
    value_name: str,
    xlabel: str,
    title: str,
) -> None:
    """Plot maximum probability histograms for each available method."""

    fig, axes = make_axes_grid()
    for axis_index, method_key in enumerate(METHOD_ORDER):
        ax = axes[axis_index]
        method_name = METHODS[method_key]
        method_frame = frame.loc[frame["method"] == method_name]

        sns.histplot(
            data=method_frame,
            x=value_name,
            hue="distribution",
            hue_order=DISTRIBUTION_ORDER,
            palette=distribution_palette,
            bins=probability_bins,
            common_bins=True,
            stat="percent",
            common_norm=False,
            ax=ax,
        )
        ax.set_title(method_name)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Percent")
        legend = ax.get_legend()
        if legend is not None:
            legend.set_title("Dataset group")

    fig.suptitle(title, fontsize=13)


teacher_max_probability_group_frame = build_max_probability_group_frame(
    probability_key="teacher_probs",
    value_name="teacher_max_probability",
)
student_max_probability_group_frame = build_max_probability_group_frame(
    probability_key="student_probs",
    value_name="student_max_probability",
)

plot_max_probability_histograms(
    teacher_max_probability_group_frame,
    value_name="teacher_max_probability",
    xlabel="Teacher maximum probability",
    title="Histogram of Teacher Maximum Probabilities",
)
plot_max_probability_histograms(
    student_max_probability_group_frame,
    value_name="student_max_probability",
    xlabel="Student maximum probability",
    title="Histogram of Student Maximum Probabilities",
)


In [ ]:
fig, axes = make_axes_grid()
for axis_index, method_key in enumerate(METHOD_ORDER):
    ax = axes[axis_index]
    method_name = METHODS[method_key]
    method_frame = max_probability_difference_group_frame.loc[
        max_probability_difference_group_frame["method"] == method_name
    ]

    sns.histplot(
        data=method_frame,
        x="max_probability_difference",
        hue="distribution",
        hue_order=DISTRIBUTION_ORDER,
        palette=distribution_palette,
        bins=40,
        kde=True,
        stat="percent",
        common_norm=False,
        ax=ax,
    )
    ax.set_title(method_name)
    ax.set_xlabel("max_proba_teacher - max_proba_student")
    ax.set_ylabel("Percent")
    legend = ax.get_legend()
    if legend is not None:
        legend.set_title("Dataset group")

fig.suptitle(
    "Histogram of Teacher-Student Maximum Probability Differences",
    fontsize=13,
)

fig, axes = make_axes_grid()
for axis_index, method_key in enumerate(METHOD_ORDER):
    ax = axes[axis_index]
    method_name = METHODS[method_key]
    method_frame = student_teacher_kl_divergence_group_frame.loc[
        student_teacher_kl_divergence_group_frame["method"] == method_name
    ]

    sns.histplot(
        data=method_frame,
        x="student_teacher_kl_divergence",
        hue="distribution",
        hue_order=DISTRIBUTION_ORDER,
        palette=distribution_palette,
        bins=40,
        kde=True,
        stat="percent",
        common_norm=False,
        ax=ax,
    )
    ax.set_title(method_name)
    ax.set_xlabel("KL(teacher || student)")
    ax.set_ylabel("Percent")
    legend = ax.get_legend()
    if legend is not None:
        legend.set_title("Dataset group")

fig.suptitle(
    "Histogram of Student-Teacher KL Divergence",
    fontsize=13,
)


In [ ]:
fig, axes = make_axes_grid()
for axis_index, method_key in enumerate(METHOD_ORDER):
    ax = axes[axis_index]
    method_name = METHODS[method_key]
    method_frame = logit_l2_distance_group_frame.loc[
        logit_l2_distance_group_frame["method"] == method_name
    ]

    sns.histplot(
        data=method_frame,
        x="logit_l2_distance",
        hue="distribution",
        hue_order=DISTRIBUTION_ORDER,
        palette=distribution_palette,
        bins=40,
        kde=True,
        stat="percent",
        common_norm=False,
        ax=ax,
    )
    ax.set_title(method_name)
    ax.set_xlabel("L2 distance between centered teacher and student logits")
    ax.set_ylabel("Percent")
    legend = ax.get_legend()
    if legend is not None:
        legend.set_title("Dataset group")

fig.suptitle(
    "Histogram of Teacher-Student Centered Logit L2 Distances",
    fontsize=13,
)


In [ ]:
fig, axes = plt.subplots(
    len(METHOD_ORDER),
    len(DISTRIBUTION_ORDER),
    figsize=(21, 4.5 * len(METHOD_ORDER)),
    constrained_layout=True,
    squeeze=False,
)
distribution_cmaps = {
    distribution: sns.light_palette(
        distribution_palette[distribution],
        as_cmap=True,
    )
    for distribution in DISTRIBUTION_ORDER
}

for row_index, method_key in enumerate(METHOD_ORDER):
    for col_index, distribution in enumerate(DISTRIBUTION_ORDER):
        ax = axes[row_index, col_index]
        class_names = DISTRIBUTION_CLASS_NAMES[distribution]
        heatmap = grouped_argmax_heatmaps.get((method_key, distribution))
        if heatmap is None:
            ax.set_visible(False)
            continue

        sns.heatmap(
            heatmap,
            ax=ax,
            cmap=distribution_cmaps[distribution],
            vmin=0,
            vmax=100,
            xticklabels=class_names,
            yticklabels=class_names,
            cbar=True,
            annot=True,
        )
        ax.set_title(f"{METHODS[method_key]} | {distribution}")
        ax.set_xlabel("Student argmax class")
        ax.set_ylabel("Teacher argmax class")
        ax.tick_params(axis="x", rotation=45)
        ax.tick_params(axis="y", rotation=0)

fig.suptitle(
    "Teacher vs Student Argmax Agreement Heatmap (%)",
    fontsize=13,
)


## ROC-AUC metrics

Each table cell reports `ROC-AUC/FPR@95TPR`.


In [ ]:
from IPython.display import Markdown, display

from distill_ood_detection.evaluation.ood_metrics import ood_detection_metrics

ID_DATASET_LABELS = {
    "cifar10_test": "CIFAR-10",
}
OOD_DATASET_LABELS = {
    "mnist_test": "Far-OOD: MNIST",
    "svhn_test": "Far-OOD: SVHN",
    "cifar100_test": "Near-OOD: CIFAR-100",
}
TEACHER_SCORE_NAME = "Teacher MSP"
STUDENT_MODEL_NAME = "MLP"
OOD_SCORE_FUNCTIONS = {
    "Max probability difference": max_probability_difference,
    "Absolute max probability difference": absolute_max_probability_difference,
    "Student-teacher KL divergence": student_teacher_kl_divergence,
}
LOGIT_L2_SCORE_NAME = "Centered logit L2 distance"
SIGNED = True


def teacher_msp_scores(artifact: dict[str, object]) -> np.ndarray:
    """Compute maximum softmax probability scores from one teacher artifact."""

    return to_numpy(artifact["probabilities"]).max(axis=1)


def build_ood_metric_table() -> pd.DataFrame:
    """Build a ROC-AUC/FPR@95 summary table for each OOD score variant."""

    id_dataset_keys = [
        dataset_key for dataset_key in ID_DATASET_LABELS if dataset_key in DATASET_ORDER
    ]
    ood_dataset_keys = [
        dataset_key for dataset_key in OOD_DATASET_LABELS if dataset_key in DATASET_ORDER
    ]

    column_index = pd.MultiIndex.from_tuples(
        [
            (ID_DATASET_LABELS[id_dataset_key], OOD_DATASET_LABELS[ood_dataset_key])
            for id_dataset_key in id_dataset_keys
            for ood_dataset_key in ood_dataset_keys
        ],
        names=["ID dataset", "OOD dataset"],
    )
    row_index = pd.MultiIndex.from_tuples(
        [("Teacher", "-", TEACHER_SCORE_NAME)]
        + [
            (STUDENT_MODEL_NAME, METHODS[method_key], score_name)
            for method_key in METHOD_ORDER
            for score_name in [*OOD_SCORE_FUNCTIONS, LOGIT_L2_SCORE_NAME]
        ],
        names=["Model", "Distillation method", "OOD score"],
    )
    metric_rows: dict[tuple[str, str, str], dict[tuple[str, str], str]] = {
        row_key: {} for row_key in row_index
    }

    for id_dataset_key in id_dataset_keys:
        id_teacher = teacher_probabilities[id_dataset_key]
        id_teacher_logits = to_numpy(id_teacher["logits"])
        id_teacher_probs = to_numpy(id_teacher["probabilities"])
        id_teacher_msp_scores = teacher_msp_scores(id_teacher)

        for ood_dataset_key in ood_dataset_keys:
            ood_teacher = teacher_probabilities[ood_dataset_key]
            ood_teacher_logits = to_numpy(ood_teacher["logits"])
            ood_teacher_probs = to_numpy(ood_teacher["probabilities"])
            ood_teacher_msp_scores = teacher_msp_scores(ood_teacher)

            labels = np.concatenate(
                [
                    np.ones(id_teacher_msp_scores.shape[0], dtype=int),
                    np.zeros(ood_teacher_msp_scores.shape[0], dtype=int),
                ]
            )
            column_key = (
                ID_DATASET_LABELS[id_dataset_key],
                OOD_DATASET_LABELS[ood_dataset_key],
            )

            teacher_metrics = ood_detection_metrics(
                labels,
                np.concatenate([id_teacher_msp_scores, ood_teacher_msp_scores]),
            )
            metric_rows[("Teacher", "-", TEACHER_SCORE_NAME)][column_key] = (
                f"{teacher_metrics['roc_auc']:.2f}/{teacher_metrics['fpr_at_95_tpr']:.2f}"
            )

            for method_key in METHOD_ORDER:
                id_student = student_probabilities[(id_dataset_key, method_key, CHECKPOINT)]
                ood_student = student_probabilities[(ood_dataset_key, method_key, CHECKPOINT)]
                id_student_logits = to_numpy(id_student["logits"])
                ood_student_logits = to_numpy(ood_student["logits"])
                id_student_probs = to_numpy(id_student["probabilities"])
                ood_student_probs = to_numpy(ood_student["probabilities"])

                for score_name, score_function in OOD_SCORE_FUNCTIONS.items():
                    id_scores = score_function(id_teacher_probs, id_student_probs, signed=SIGNED)
                    ood_scores = score_function(ood_teacher_probs, ood_student_probs, signed=SIGNED)
                    metrics = ood_detection_metrics(
                        labels,
                        np.concatenate([id_scores, ood_scores]),
                    )
                    metric_rows[
                        (STUDENT_MODEL_NAME, METHODS[method_key], score_name)
                    ][column_key] = f"{metrics['roc_auc']:.2f}/{metrics['fpr_at_95_tpr']:.2f}"

                id_scores = logit_l2_distance(
                    id_teacher_logits,
                    id_student_logits,
                    signed=SIGNED,
                )
                ood_scores = logit_l2_distance(
                    ood_teacher_logits,
                    ood_student_logits,
                    signed=SIGNED,
                )
                metrics = ood_detection_metrics(
                    labels,
                    np.concatenate([id_scores, ood_scores]),
                )
                metric_rows[
                    (STUDENT_MODEL_NAME, METHODS[method_key], LOGIT_L2_SCORE_NAME)
                ][column_key] = f"{metrics['roc_auc']:.2f}/{metrics['fpr_at_95_tpr']:.2f}"

    metric_table = pd.DataFrame.from_dict(
        metric_rows,
        orient="index",
        columns=column_index,
    )
    metric_table.index = row_index
    return metric_table

roc_auc_metric_table = build_ood_metric_table()
roc_auc_metric_table

In [ ]:
from sklearn.metrics import roc_curve


def build_single_ood_roc_curve(
    labels: np.ndarray,
    scores: np.ndarray,
    *,
    ood_dataset_label: str,
) -> dict[str, object]:
    """Build one ROC curve and its FPR@95TPR marker point."""

    fpr, tpr, _ = roc_curve(labels, scores, pos_label=1)
    metrics = ood_detection_metrics(labels, scores)
    candidate_indices = np.flatnonzero(tpr >= 0.95)
    marker_index = candidate_indices[np.argmin(fpr[candidate_indices])]

    return {
        "fpr": fpr,
        "tpr": tpr,
        "fpr_at_95_tpr": metrics["fpr_at_95_tpr"],
        "tpr_at_95_marker": float(tpr[marker_index]),
        "roc_auc": metrics["roc_auc"],
        "ood_dataset_label": ood_dataset_label,
        "legend_label": (
            f"{ood_dataset_label} "
            f"({metrics['roc_auc']:.4f}/{metrics['fpr_at_95_tpr']:.4f})"
        ),
    }


def build_student_ood_roc_curves() -> dict[tuple[str, str], list[dict[str, object]]]:
    """Build ROC curve data grouped by student method and OOD score."""

    score_names = [TEACHER_SCORE_NAME, *OOD_SCORE_FUNCTIONS, LOGIT_L2_SCORE_NAME]
    id_dataset_keys = [
        dataset_key for dataset_key in ID_DATASET_LABELS if dataset_key in DATASET_ORDER
    ]
    ood_dataset_keys = [
        dataset_key for dataset_key in OOD_DATASET_LABELS if dataset_key in DATASET_ORDER
    ]
    curves_by_axis = {
        (method_key, score_name): []
        for method_key in METHOD_ORDER
        for score_name in score_names
    }

    for id_dataset_key in id_dataset_keys:
        id_teacher = teacher_probabilities[id_dataset_key]
        id_teacher_logits = to_numpy(id_teacher["logits"])
        id_teacher_probs = to_numpy(id_teacher["probabilities"])
        id_teacher_msp_scores = teacher_msp_scores(id_teacher)

        for ood_dataset_key in ood_dataset_keys:
            ood_teacher = teacher_probabilities[ood_dataset_key]
            ood_teacher_logits = to_numpy(ood_teacher["logits"])
            ood_teacher_probs = to_numpy(ood_teacher["probabilities"])
            ood_teacher_msp_scores = teacher_msp_scores(ood_teacher)
            labels = np.concatenate(
                [
                    np.ones(id_teacher_probs.shape[0], dtype=int),
                    np.zeros(ood_teacher_probs.shape[0], dtype=int),
                ]
            )

            for method_key in METHOD_ORDER:
                curves_by_axis[(method_key, TEACHER_SCORE_NAME)].append(
                    build_single_ood_roc_curve(
                        labels,
                        np.concatenate([id_teacher_msp_scores, ood_teacher_msp_scores]),
                        ood_dataset_label=OOD_DATASET_LABELS[ood_dataset_key],
                    )
                )

                id_student = student_probabilities[(id_dataset_key, method_key, CHECKPOINT)]
                ood_student = student_probabilities[(ood_dataset_key, method_key, CHECKPOINT)]
                id_student_logits = to_numpy(id_student["logits"])
                ood_student_logits = to_numpy(ood_student["logits"])
                id_student_probs = to_numpy(id_student["probabilities"])
                ood_student_probs = to_numpy(ood_student["probabilities"])

                for score_name, score_function in OOD_SCORE_FUNCTIONS.items():
                    id_scores = score_function(id_teacher_probs, id_student_probs, signed=SIGNED)
                    ood_scores = score_function(ood_teacher_probs, ood_student_probs, signed=SIGNED)
                    curves_by_axis[(method_key, score_name)].append(
                        build_single_ood_roc_curve(
                            labels,
                            np.concatenate([id_scores, ood_scores]),
                            ood_dataset_label=OOD_DATASET_LABELS[ood_dataset_key],
                        )
                    )

                id_scores = logit_l2_distance(
                    id_teacher_logits,
                    id_student_logits,
                    signed=SIGNED,
                )
                ood_scores = logit_l2_distance(
                    ood_teacher_logits,
                    ood_student_logits,
                    signed=SIGNED,
                )
                curves_by_axis[(method_key, LOGIT_L2_SCORE_NAME)].append(
                    build_single_ood_roc_curve(
                        labels,
                        np.concatenate([id_scores, ood_scores]),
                        ood_dataset_label=OOD_DATASET_LABELS[ood_dataset_key],
                    )
                )

    return curves_by_axis


roc_curves_by_axis = build_student_ood_roc_curves()
score_names = [TEACHER_SCORE_NAME, *OOD_SCORE_FUNCTIONS, LOGIT_L2_SCORE_NAME]
fig, axes = plt.subplots(
    len(METHOD_ORDER),
    len(score_names),
    figsize=(7 * len(score_names), 4.8 * len(METHOD_ORDER)),
    constrained_layout=True,
    squeeze=False,
)
ood_palette = dict(
    zip(
        OOD_DATASET_LABELS.values(),
        sns.color_palette("deep", n_colors=len(OOD_DATASET_LABELS)),
        strict=True,
    )
)

for row_index, method_key in enumerate(METHOD_ORDER):
    for col_index, score_name in enumerate(score_names):
        ax = axes[row_index, col_index]
        curves = roc_curves_by_axis[(method_key, score_name)]

        for curve in curves:
            color = ood_palette[curve["ood_dataset_label"]]
            ax.plot(
                curve["fpr"],
                curve["tpr"],
                color=color,
                linewidth=1.8,
                alpha=0.85,
                label=curve["legend_label"],
            )
            ax.scatter(
                curve["fpr_at_95_tpr"],
                curve["tpr_at_95_marker"],
                color=color,
                edgecolor="black",
                linewidth=0.5,
                s=38,
                zorder=3,
            )

        ax.plot([0, 1], [0, 1], color="black", linestyle="--", linewidth=1, alpha=0.45)
        ax.axhline(0.95, color="black", linestyle=":", linewidth=1, alpha=0.7)
        ax.set_title(f"{METHODS[method_key]} | {score_name}")
        ax.set_xlabel("False positive rate")
        ax.set_ylabel("True positive rate")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1.01)
        ax.legend(title="OOD dataset (ROC-AUC/FPR@95)", fontsize=12, title_fontsize=12)

fig.suptitle("Student OOD ROC Curves", fontsize=13)
